# 오답노트 · 손글씨 지우기 AI 학습

**실행 방법**
1. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 저장
2. **런타임 → 모두 실행** (Ctrl+F9)
3. 약 40~60분 뒤 마지막 칸에서 `inkseg.onnx` 파일이 자동으로 다운로드됩니다
4. 받은 파일을 PC의 `wrong_question/train/` 폴더에 넣어주세요

중간에 창을 닫으면 학습이 멈춥니다. 탭은 열어둔 채로 다른 일을 하셔도 됩니다.

In [ ]:
!nvidia-smi -L || echo "GPU가 없습니다 — 런타임 유형을 T4 GPU로 바꿔주세요"

## 1. 코드 준비

In [ ]:
!mkdir -p wn work

In [ ]:
%%writefile wn/__init__.py
# 오답노트 손글씨 분리 모델 학습 패키지

In [ ]:
%%writefile wn/normalize.py
"""배경 정리 (앱의 app/js/clean-worker.js estimateBackground 와 같은 방식).

모델은 항상 '배경이 정리된' 이미지를 입력으로 받습니다.
스캔본(학습 데이터)과 폰 사진(실제 사용)의 조명 차이를 줄이기 위해서입니다.
"""
import numpy as np
from PIL import Image
from scipy.ndimage import maximum_filter, uniform_filter


def estimate_background(img: np.ndarray) -> np.ndarray:
    """img: HxWx3 uint8 → 같은 크기의 배경(종이) 추정치 float32."""
    h, w = img.shape[:2]
    max_side = max(h, w)
    f = max(1, round(max_side / 320))
    lh, lw = -(-h // f), -(-w // f)
    pad = np.pad(img.astype(np.float32), ((0, lh * f - h), (0, lw * f - w), (0, 0)), mode='edge')
    low = pad.reshape(lh, f, lw, f, 3).mean(axis=(1, 3))

    r = max(2, round(max(40, max_side * 0.04) / f))
    rb = max(1, r >> 1)
    bg = np.empty_like(low)
    for c in range(3):
        a = maximum_filter(low[..., c], size=2 * r + 1, mode='nearest')
        a = uniform_filter(a, size=2 * rb + 1, mode='nearest')
        a = uniform_filter(a, size=2 * rb + 1, mode='nearest')
        bg[..., c] = a
    up = np.stack([
        np.asarray(Image.fromarray(bg[..., c]).resize((w, h), Image.BILINEAR))
        for c in range(3)
    ], axis=-1)
    return np.maximum(up, 1.0)


def normalize(img: np.ndarray) -> np.ndarray:
    """종이를 흰색으로, 그림자 제거. 앱의 renderBase()와 같은 결과(근사)."""
    bg = estimate_background(img)
    out = img.astype(np.float32) / bg * 255.0
    out = np.minimum(out, 255.0) * 1.08
    return np.clip(out, 0, 255).astype(np.uint8)

In [ ]:
%%writefile wn/model.py
"""작은 U-Net: 배경 정리된 RGB → 손글씨일 확률(픽셀별).

약 60만 개 파라미터 (fp16 기준 1~2MB) — 아이패드/안드로이드 브라우저에서 돌리기 위해 작게 설계.
입력 크기는 16의 배수여야 합니다 (앱에서 여백을 채워 맞춤).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class InkUNet(nn.Module):
    def __init__(self, chs=(16, 32, 64, 96, 128)):
        super().__init__()
        self.enc = nn.ModuleList()
        cin = 3
        for c in chs:
            self.enc.append(ConvBlock(cin, c))
            cin = c
        self.dec = nn.ModuleList()
        for i in range(len(chs) - 1, 0, -1):
            self.dec.append(ConvBlock(chs[i] + chs[i - 1], chs[i - 1]))
        self.head = nn.Conv2d(chs[0], 1, 1)

    def forward(self, x):
        # x: (N,3,H,W) 0~1
        x = x - 0.5
        skips = []
        for i, block in enumerate(self.enc):
            if i > 0:
                x = F.max_pool2d(x, 2)
            x = block(x)
            skips.append(x)
        skips.pop()
        for block in self.dec:
            s = skips.pop()
            x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
            x = block(torch.cat([x, s], dim=1))
        return self.head(x)  # 로짓


def count_params(m):
    return sum(p.numel() for p in m.parameters())

In [ ]:
%%writefile wn/synth.py
"""학습 데이터 합성.

한 장의 학습 샘플 = (입력: 배경 정리된 사진 조각, 정답: 손글씨 위치 마스크)

재료
  1) 인쇄 페이지
     - 한국 문제집 스타일로 직접 그린 페이지 (명조/고딕, 문제 번호, 상자, 답칸, 색 띠, 도형,
       연한 회색 인쇄 등 — 지금 규칙 방식이 실수하던 요소를 일부러 많이 넣음)
     - ExamInk-Seg 의 '손글씨 지운' 실제 시험지 스캔 (target)
  2) 손글씨 층
     - 한글 손글씨 폰트로 쓴 풀이/답 (숫자, 식, 한글, 영어)
     - 손으로 그은 선: 동그라미, 밑줄, 체크, 화살표, 낙서, 계산 흔적
     - ExamInk-Seg 실제 학생 손글씨를 뽑아낸 것
  3) 필기구 스타일: 연필(회색·거친 결·연함~진함), 볼펜(검정/파랑), 빨간펜, 색연필
  4) 폰 사진 흉내: 조명 얼룩, 색온도, 흐림, 노이즈, JPEG
그 뒤 앱과 똑같이 배경 정리(normalize)를 거쳐 입력을 만듭니다.
"""
import io
import math
import os
import random
import urllib.request

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont
from scipy.ndimage import gaussian_filter, binary_opening, binary_dilation

from .normalize import normalize

# ---------------------------------------------------------------- 폰트
FONT_BASE = 'https://github.com/google/fonts/raw/main/ofl/'
PRINT_FONTS = [
    'nanummyeongjo/NanumMyeongjo-Regular.ttf',
    'nanummyeongjo/NanumMyeongjo-Bold.ttf',
    'nanumgothic/NanumGothic-Regular.ttf',
    'nanumgothic/NanumGothic-Bold.ttf',
    'notosanskr/NotoSansKR%5Bwght%5D.ttf',
    'notoserifkr/NotoSerifKR%5Bwght%5D.ttf',
]
HAND_FONTS = [
    'nanumpenscript/NanumPenScript-Regular.ttf',
    'nanumbrushscript/NanumBrushScript-Regular.ttf',
    'gaegu/Gaegu-Regular.ttf',
    'gaegu/Gaegu-Bold.ttf',
    'himelody/HiMelody-Regular.ttf',
    'gamjaflower/GamjaFlower-Regular.ttf',
    'poorstory/PoorStory-Regular.ttf',
    'yeonsung/YeonSung-Regular.ttf',
]


def ensure_fonts(font_dir):
    os.makedirs(font_dir, exist_ok=True)
    for rel in PRINT_FONTS + HAND_FONTS:
        dst = os.path.join(font_dir, os.path.basename(rel).replace('%5B', '[').replace('%5D', ']'))
        if not os.path.exists(dst):
            urllib.request.urlretrieve(FONT_BASE + rel, dst)


def _font_paths(font_dir, rels):
    return [os.path.join(font_dir, os.path.basename(r).replace('%5B', '[').replace('%5D', ']')) for r in rels]


class Fonts:
    def __init__(self, font_dir):
        ensure_fonts(font_dir)
        self.print_paths = _font_paths(font_dir, PRINT_FONTS)
        self.hand_paths = _font_paths(font_dir, HAND_FONTS)
        self._cache = {}

    def get(self, path, size):
        key = (path, size)
        if key not in self._cache:
            self._cache[key] = ImageFont.truetype(path, size)
        return self._cache[key]

    def print_font(self, rng, size, bold=None):
        paths = self.print_paths
        if bold is True:
            paths = [p for p in paths if 'Bold' in p or 'wght' in p]
        return self.get(rng.choice(paths), size)

    def hand_font(self, rng, size):
        return self.get(rng.choice(self.hand_paths), size)


# ---------------------------------------------------------------- 문장 재료
KO_PHRASES = [
    '주어진 비를 간단한 자연수의 비로 나타내면', '안에 알맞은 수를 구해 보시오.', '다음 중 옳은 것을 고르시오.',
    '가로와 세로의 비가', '직사각형이 있습니다.', '세로는 몇 cm입니까?', '넓이를 구하시오.', '비례식을 만들고 답을 구해 보시오.',
    '우리말과 일치하도록 주어진 말을 이용하여 문장을 완성하시오.', '구슬의 수의 비는', '가지고 있다면', '몇 개입니까?',
    '케이크 한 개를 나누어 먹었습니다.', '전체의', '먹었습니다.', '물음에 답하시오.', '그림을 보고', '설명한 것입니다.',
    '알맞은 말에 ○표 하시오.', '계산 결과를 비교하여', '빈칸에 알맞은 수를 써넣으시오.', '분수의 나눗셈', '소수의 곱셈',
    '원의 넓이', '삼각형의 둘레', '평균을 구하시오.', '비율을 백분율로 나타내시오.', '그래프를 보고', '규칙을 찾아',
    '어느 것입니까?', '바르게 고친 것은?', '틀린 이유를 쓰시오.', '식을 세워 보시오.', '단원 평가', '교과서 pick',
    '서술형 대비 연습문제', '다음 문장을 읽고', '그들은 해변에 갔다.', '물이 100도에 끓으니?', '아니, 그녀는 머핀을 먹었어.',
]
EN_WORDS = ['live', 'boil', 'have', 'go', 'Heidi', 'in', 'New', 'York', 'last', 'year', 'She', 'Sydney', 'now',
            'water', 'at', 'They', 'to', 'the', 'mountains', 'beach', 'Amy', 'doughnut', 'for', 'No', "didn't",
            'muffin', 'Did', 'Does', 'Yes', 'it', 'wrote', 'Christmas', 'cards', 'my', 'aunt', 'teaches', 'music']
HAND_TEXTS = [
    '12×5/9=25×△', '17/15=△', '17÷15=△', '△=4', '6:9=2:3', '1:2=12:□', '5:3=25:□', '24cm', '15개', '9개',
    '가, 라', '답', '1:20', '4/15', '20/3×1/25', '12', '25', '3', '7', '0.5', '3.14', '100', '=', '+', '-', '×',
    'lived', 'lives', 'Does', 'boil', 'it', 'does', "didn't", 'go', 'went', 'Did', 'have', 'ate', 'I wrote',
    '모르겠다', '다시', '왜?', '정답', '풀이', '①', '②', '③', '④', '⑤', '○', '×', '?', '!!', '1/2', '2/3', '3/8', '9/16',
]


def _rand_ko_line(rng, max_chars=26):
    s = rng.choice(KO_PHRASES)
    while len(s) < max_chars * 0.6 and rng.random() < 0.7:
        s += ' ' + rng.choice(KO_PHRASES + [str(rng.randint(1, 99)), f'{rng.randint(1, 30)} : {rng.randint(1, 30)}'])
    return s[:max_chars]


def _rand_en_line(rng):
    return ' '.join(rng.choice(EN_WORDS) for _ in range(rng.randint(3, 8)))


# ---------------------------------------------------------------- 인쇄 페이지
def _gray(v):
    return (v, v, v)


def render_print_page(rng, fonts, w=960, h=960):
    paper = rng.randint(238, 255)
    img = Image.new('RGB', (w, h), (paper, paper, max(225, paper - rng.randint(0, 10))))
    d = ImageDraw.Draw(img)
    base = rng.randint(22, 36)              # 본문 글자 크기
    ink = _gray(rng.randint(10, 45))        # 본문 인쇄 색
    light = _gray(rng.randint(95, 175))     # 연한 회색 인쇄 (지우면 안 되는 것)
    accent = rng.choice([(232, 96, 80), (240, 120, 90), (90, 170, 110), (120, 180, 140), (80, 130, 200), (230, 90, 140)])
    y = rng.randint(-base, base)
    x0 = rng.randint(10, 60)

    while y < h - base:
        kind = rng.choices(
            ['problem', 'text', 'box', 'answer', 'paren', 'band', 'label', 'figure', 'gray', 'dots', 'table', 'english'],
            weights=[18, 14, 8, 10, 8, 4, 6, 6, 8, 3, 3, 8])[0]
        fs = int(base * rng.uniform(0.85, 1.15))
        if kind == 'problem':
            num = str(rng.randint(1, 30))
            d.text((x0, y), num, font=fonts.print_font(rng, int(fs * 1.5), bold=True), fill=ink)
            tx = x0 + int(fs * 1.4)
            for _ in range(rng.randint(1, 3)):
                d.text((tx, y), _rand_ko_line(rng), font=fonts.print_font(rng, fs), fill=ink)
                y += int(fs * 1.6)
        elif kind == 'text':
            d.text((x0 + rng.randint(0, 80), y), _rand_ko_line(rng), font=fonts.print_font(rng, fs), fill=ink)
            y += int(fs * 1.6)
        elif kind == 'english':
            d.text((x0 + rng.randint(0, 120), y), _rand_en_line(rng), font=fonts.print_font(rng, fs), fill=ink)
            y += int(fs * 1.6)
        elif kind == 'gray':
            d.text((x0 + rng.randint(0, 100), y), _rand_ko_line(rng), font=fonts.print_font(rng, int(fs * 0.85)), fill=light)
            y += int(fs * 1.4)
        elif kind == 'box':
            bw, bh = rng.randint(w // 3, w - 2 * x0), int(fs * rng.uniform(2.2, 4.5))
            bx = rng.randint(x0, max(x0 + 1, w - bw - 10))
            col = rng.choice([ink, light, _gray(rng.randint(60, 130))])
            d.rounded_rectangle((bx, y, bx + bw, y + bh), radius=rng.randint(4, 18), outline=col, width=rng.randint(1, 3))
            # 상자 안 분수/식
            cx = bx + bw // 3
            fnt = fonts.print_font(rng, fs)
            d.text((cx, y + bh * 0.15), str(rng.randint(1, 9)), font=fnt, fill=ink)
            d.line((cx - 4, y + bh / 2, cx + fs, y + bh / 2), fill=ink, width=2)
            d.text((cx, y + bh * 0.52), str(rng.randint(2, 19)), font=fnt, fill=ink)
            d.text((cx + fs * 2, y + bh * 0.3), ':', font=fnt, fill=ink)
            d.rectangle((cx + fs * 3, y + bh * 0.12, cx + fs * 3.8, y + bh * 0.42), outline=ink, width=2)
            y += bh + int(fs * 0.8)
        elif kind == 'answer':
            fnt = fonts.print_font(rng, fs, bold=rng.random() < 0.5)
            lab = rng.choice(['식 |', '답 |', '→', '답:'])
            lx = x0 + rng.randint(0, w // 3)
            d.text((lx, y), lab, font=fnt, fill=ink)
            ly = y + int(fs * 1.2)
            d.line((lx + fs * 1.5, ly, min(w - 10, lx + rng.randint(w // 3, w)), ly), fill=rng.choice([ink, light]), width=rng.randint(1, 2))
            y += int(fs * 2.2)
        elif kind == 'paren':
            fnt = fonts.print_font(rng, fs)
            px = x0 + rng.randint(w // 5, w // 2)
            d.text((px, y), '(', font=fnt, fill=ink)
            d.text((min(w - 30, px + rng.randint(w // 4, w // 2)), y), ')', font=fnt, fill=ink)
            y += int(fs * 1.8)
        elif kind == 'band':
            bh = int(fs * 1.8)
            col = tuple(int(c + (255 - c) * rng.uniform(0.0, 0.6)) for c in accent)
            d.rounded_rectangle((rng.randint(-40, x0), y, w + 40, y + bh), radius=bh // 2, fill=col)
            if rng.random() < 0.6:
                d.ellipse((x0, y + 3, x0 + bh - 6, y + bh - 3), fill=accent)
                d.text((x0 + bh * 0.3, y + bh * 0.15), str(rng.randint(1, 9)), font=fonts.print_font(rng, int(fs * 1.1), bold=True), fill=(255, 255, 255))
            d.text((x0 + bh * 1.2, y + bh * 0.15), _rand_ko_line(rng, 14), font=fonts.print_font(rng, int(fs * 1.1), bold=True), fill=ink)
            y += bh + int(fs * 0.8)
        elif kind == 'label':
            fnt = fonts.print_font(rng, fs)
            lw = int(fs * rng.uniform(4, 7))
            col = tuple(int(c + (255 - c) * rng.uniform(0.2, 0.6)) for c in accent)
            d.rectangle((x0, y, x0 + lw, y + int(fs * 1.6)), fill=col)
            d.text((x0 + lw * 0.3, y + fs * 0.2), rng.choice(EN_WORDS), font=fnt, fill=_gray(rng.randint(30, 80)))
            d.text((x0 + lw + fs, y + fs * 0.2), _rand_en_line(rng), font=fnt, fill=ink)
            y += int(fs * 2.2)
        elif kind == 'figure':
            fx = rng.randint(x0, w // 2)
            fw, fh = rng.randint(80, 220), rng.randint(80, 200)
            col = rng.choice([ink, (150, 100, 60), light, accent])
            shape = rng.choice(['rect', 'tri', 'circle'])
            wid = rng.randint(2, 8)
            if shape == 'rect':
                d.rectangle((fx, y, fx + fw, y + fh), outline=col, width=wid)
                if rng.random() < 0.5:
                    d.rectangle((fx + wid, y + wid, fx + fw - wid, y + fh - wid), fill=(205, 220, 240))
            elif shape == 'tri':
                d.polygon([(fx, y + fh), (fx + fw, y + fh), (fx + fw * rng.random(), y)], outline=col, width=wid)
            else:
                d.ellipse((fx, y, fx + fh, y + fh), outline=col, width=wid)
            d.text((fx + fw + 8, y + fh // 2), f'{rng.randint(2, 30)} cm', font=fonts.print_font(rng, int(fs * 0.9)), fill=ink)
            y += fh + int(fs)
        elif kind == 'dots':
            for xx in range(x0, w - x0, 8):
                d.point((xx, y + fs // 2), fill=light)
            y += fs
        elif kind == 'table':
            rows, cols = rng.randint(2, 4), rng.randint(3, 6)
            cw, ch = (w - 2 * x0) // cols, int(fs * 1.6)
            for r in range(rows + 1):
                d.line((x0, y + r * ch, x0 + cols * cw, y + r * ch), fill=ink, width=1)
            for c in range(cols + 1):
                d.line((x0 + c * cw, y, x0 + c * cw, y + rows * ch), fill=ink, width=1)
            for r in range(rows):
                for c in range(cols):
                    if rng.random() < 0.6:
                        d.text((x0 + c * cw + 8, y + r * ch + 4), str(rng.randint(1, 99)), font=fonts.print_font(rng, int(fs * 0.85)), fill=ink)
            y += rows * ch + fs
        y += rng.randint(0, fs)
    return np.asarray(img)


# ---------------------------------------------------------------- 손글씨 층
def _bezier(p0, p1, p2, n=24):
    return [((1 - t) ** 2 * p0[0] + 2 * (1 - t) * t * p1[0] + t * t * p2[0],
             (1 - t) ** 2 * p0[1] + 2 * (1 - t) * t * p1[1] + t * t * p2[1]) for t in np.linspace(0, 1, n)]


def _draw_strokes(rng, d, w, h, scale):
    kind = rng.choice(['circle', 'underline', 'check', 'arrow', 'scribble', 'cross', 'bracket', 'calc_lines'])
    lw = max(1, int(rng.uniform(1.5, 4.5) * scale))
    cx, cy = rng.uniform(0, w), rng.uniform(0, h)
    s = rng.uniform(30, 160) * scale
    if kind == 'circle':
        pts = [(cx + s * math.cos(t) * rng.uniform(0.9, 1.1), cy + s * 0.6 * math.sin(t) * rng.uniform(0.9, 1.1))
               for t in np.linspace(rng.uniform(0, 1), rng.uniform(6.0, 7.0), 40)]
        d.line(pts, fill=255, width=lw, joint='curve')
    elif kind == 'underline':
        pts = _bezier((cx, cy), (cx + s, cy + rng.uniform(-8, 8) * scale), (cx + 2 * s, cy + rng.uniform(-6, 6) * scale))
        d.line(pts, fill=255, width=lw, joint='curve')
    elif kind == 'check':
        d.line([(cx, cy), (cx + s * 0.3, cy + s * 0.4), (cx + s * 0.8, cy - s * 0.5)], fill=255, width=lw, joint='curve')
    elif kind == 'arrow':
        ex, ey = cx + rng.uniform(-s, s), cy + rng.uniform(-s, s)
        d.line([(cx, cy), (ex, ey)], fill=255, width=lw)
        a = math.atan2(ey - cy, ex - cx)
        for da in (2.6, -2.6):
            d.line([(ex, ey), (ex + 15 * scale * math.cos(a + da), ey + 15 * scale * math.sin(a + da))], fill=255, width=lw)
    elif kind == 'scribble':
        pts = [(cx, cy)]
        for _ in range(rng.randint(5, 20)):
            pts.append((pts[-1][0] + rng.uniform(-s, s) * 0.3, pts[-1][1] + rng.uniform(-s, s) * 0.3))
        d.line(pts, fill=255, width=lw, joint='curve')
    elif kind == 'cross':
        d.line([(cx - s / 3, cy - s / 3), (cx + s / 3, cy + s / 3)], fill=255, width=lw)
        d.line([(cx - s / 3, cy + s / 3), (cx + s / 3, cy - s / 3)], fill=255, width=lw)
    elif kind == 'bracket':
        d.line([(cx, cy), (cx, cy - s * 0.4), (cx + s * 1.5, cy - s * 0.4), (cx + s * 1.5, cy)], fill=255, width=lw)
    else:  # 계산 흔적: 짧은 가로선 여러 개
        for i in range(rng.randint(2, 5)):
            yy = cy + i * s * 0.35
            d.line([(cx, yy), (cx + s * rng.uniform(0.8, 2), yy + rng.uniform(-4, 4))], fill=255, width=lw)


def _draw_hand_text(rng, fonts, layer, w, h, scale):
    text = rng.choice(HAND_TEXTS) if rng.random() < 0.7 else ' '.join(rng.choice(HAND_TEXTS) for _ in range(rng.randint(2, 4)))
    size = int(rng.uniform(26, 60) * scale)
    font = fonts.hand_font(rng, size)
    tile = Image.new('L', (int(size * (len(text) + 2) * 0.8) + 20, int(size * 1.8)), 0)
    td = ImageDraw.Draw(tile)
    x = 10
    for ch in text:   # 글자마다 위치/크기 흔들기
        f = fonts.hand_font(rng, max(8, int(size * rng.uniform(0.85, 1.15)))) if rng.random() < 0.3 else font
        td.text((x, size * 0.2 + rng.uniform(-size * 0.12, size * 0.12)), ch, font=f, fill=255)
        x += int(size * rng.uniform(0.5, 0.85)) if ch != ' ' else int(size * 0.4)
    if rng.random() < 0.5:
        tile = tile.filter(ImageFilter.MaxFilter(3))   # 굵은 필기
    tile = tile.rotate(rng.uniform(-10, 10), resample=Image.BILINEAR, expand=True)
    sx = rng.randint(-tile.width // 3, max(1, w - tile.width // 2))
    sy = rng.randint(-tile.height // 3, max(1, h - tile.height // 2))
    layer.paste(Image.fromarray(np.maximum(np.asarray(layer.crop((sx, sy, sx + tile.width, sy + tile.height))), np.asarray(tile))), (sx, sy))


def render_hand_alpha(rng, fonts, w, h, scale=1.0, n_text=None, n_strokes=None):
    """손글씨 덮개 (0~1). 2배로 그린 뒤 줄여서 부드러운 가장자리."""
    W2, H2 = w * 2, h * 2
    layer = Image.new('L', (W2, H2), 0)
    d = ImageDraw.Draw(layer)
    for _ in range(rng.randint(1, 6) if n_text is None else n_text):
        _draw_hand_text(rng, fonts, layer, W2, H2, scale * 2)
    for _ in range(rng.randint(0, 5) if n_strokes is None else n_strokes):
        _draw_strokes(rng, d, W2, H2, scale * 2)
    return np.asarray(layer.resize((w, h), Image.BILINEAR)).astype(np.float32) / 255.0


# ---------------------------------------------------------------- 필기구 스타일
INK_STYLES = ['pencil', 'pencil', 'pencil', 'pencil_dark', 'pen_black', 'pen_blue', 'pen_red', 'color_pencil']


def apply_ink(page, alpha, rng, style=None):
    """page: HxWx3 float(0~255), alpha: HxW (0~1) → (새 페이지, 실제로 보이는 진하기)"""
    style = style or rng.choice(INK_STYLES)
    h, w = alpha.shape
    if style.startswith('pencil'):
        g = rng.uniform(55, 115)
        color = np.array([g, g, g * rng.uniform(1.0, 1.08)])
        strength = rng.uniform(0.75, 1.0) if style == 'pencil_dark' else rng.uniform(0.3, 0.8)
        grain = gaussian_filter(np.random.default_rng(rng.randint(0, 2**31)).random((h, w)).astype(np.float32), 0.7)
        grain = (grain - grain.mean()) / (grain.std() + 1e-6)
        a = alpha * strength * np.clip(0.75 + 0.35 * grain, 0.15, 1.2)
        a = gaussian_filter(a, rng.uniform(0.3, 0.8))
    elif style == 'pen_black':
        color = np.array([rng.uniform(10, 40)] * 3)
        a = alpha * rng.uniform(0.85, 1.0)
    elif style == 'pen_blue':
        color = np.array([rng.uniform(20, 60), rng.uniform(40, 80), rng.uniform(120, 190)])
        a = alpha * rng.uniform(0.8, 1.0)
    elif style == 'pen_red':
        color = np.array([rng.uniform(190, 230), rng.uniform(30, 80), rng.uniform(50, 100)])
        a = alpha * rng.uniform(0.8, 1.0)
    else:  # 색연필
        base = rng.choice([(230, 90, 140), (90, 120, 220), (60, 160, 90), (240, 150, 60)])
        color = np.array(base, dtype=np.float32)
        a = alpha * rng.uniform(0.4, 0.8)
    a = np.clip(a, 0, 1)[..., None]
    out = page * (1 - a + a * color / 255.0)       # 곱하기 합성 (인쇄 위에 쓰면 더 진해짐)
    visible = (a[..., 0] * (1 - color.min() / 255.0))
    return out, visible


# ---------------------------------------------------------------- 폰 사진 흉내
def photo_aug(img, rng):
    h, w = img.shape[:2]
    out = img.astype(np.float32)
    # 조명 얼룩 (저해상도 랜덤 → 확대)
    gh, gw = rng.randint(2, 5), rng.randint(2, 5)
    field = np.random.default_rng(rng.randint(0, 2**31)).uniform(rng.uniform(0.55, 0.9), 1.05, (gh, gw)).astype(np.float32)
    field = np.asarray(Image.fromarray(field).resize((w, h), Image.BICUBIC))
    out *= field[..., None]
    # 색온도
    out *= np.array([rng.uniform(0.92, 1.06), rng.uniform(0.95, 1.03), rng.uniform(0.85, 1.05)])[None, None]
    # 흐림 / 노이즈
    if rng.random() < 0.7:
        out = gaussian_filter(out, sigma=(rng.uniform(0.2, 1.3), rng.uniform(0.2, 1.3), 0))
    out += np.random.default_rng(rng.randint(0, 2**31)).normal(0, rng.uniform(0, 5), out.shape)
    out = np.clip(out, 0, 255).astype(np.uint8)
    # JPEG
    buf = io.BytesIO()
    Image.fromarray(out).save(buf, 'JPEG', quality=rng.randint(55, 95))
    return np.asarray(Image.open(io.BytesIO(buf.getvalue())).convert('RGB'))


# ---------------------------------------------------------------- 실제 시험지 (ExamInk-Seg)
class ExamInk:
    """ExamInk-Seg 폴더: data/{train,val}/{source,target}/*.jpg"""

    def __init__(self, root, split='train'):
        self.src_dir = os.path.join(root, 'data', split, 'source')
        self.tgt_dir = os.path.join(root, 'data', split, 'target')
        names = sorted(os.listdir(self.src_dir)) if os.path.isdir(self.src_dir) else []
        self.names = [n for n in names if os.path.exists(os.path.join(self.tgt_dir, n))]

    def __len__(self):
        return len(self.names)

    def load(self, rng, scale):
        n = rng.choice(self.names)
        src = Image.open(os.path.join(self.src_dir, n)).convert('RGB')
        tgt = Image.open(os.path.join(self.tgt_dir, n)).convert('RGB')
        if tgt.size != src.size:
            tgt = tgt.resize(src.size, Image.BILINEAR)
        size = (max(64, int(src.width * scale)), max(64, int(src.height * scale)))
        src, tgt = src.resize(size, Image.BILINEAR), tgt.resize(size, Image.BILINEAR)
        return np.asarray(src), np.asarray(tgt)

    @staticmethod
    def ink_from_pair(src, tgt):
        """원본-정답 차이로 손글씨 덮개(진하기)와 마스크 계산."""
        ls = src.astype(np.float32).mean(axis=2)
        lt = tgt.astype(np.float32).mean(axis=2)
        diff = np.abs(src.astype(np.int16) - tgt.astype(np.int16)).max(axis=2)
        darker = (lt - ls)
        mask = ((diff > 40) & (darker > 12)) | (darker > 45)
        mask = binary_opening(mask, iterations=1)
        alpha = np.clip(darker / np.maximum(lt, 1) * 1.6, 0, 1) * binary_dilation(mask, iterations=1)
        return alpha.astype(np.float32), mask


# ---------------------------------------------------------------- 샘플 1개 만들기
def _random_crop(rng, arrs, size):
    h, w = arrs[0].shape[:2]
    if h < size or w < size:
        ph, pw = max(0, size - h), max(0, size - w)
        arrs = [np.pad(a, ((0, ph), (0, pw)) + ((0, 0),) * (a.ndim - 2), mode='edge') for a in arrs]
        h, w = arrs[0].shape[:2]
    y, x = rng.randint(0, h - size), rng.randint(0, w - size)
    return [a[y:y + size, x:x + size] for a in arrs]


def make_sample(rng, fonts, exam=None, tile=384, region=768):
    """반환: (입력 RGB uint8 tile×tile, 마스크 uint8 0/1)"""
    mode = rng.random()
    use_exam = exam is not None and len(exam) > 0
    scale = rng.uniform(0.75, 1.3)

    if use_exam and mode < 0.3:
        # (a) 실제 학생 손글씨가 있는 시험지 그대로
        src, tgt = exam.load(rng, scale * rng.uniform(0.75, 1.0))
        src, tgt = _random_crop(rng, [src, tgt], region)
        _, mask = ExamInk.ink_from_pair(src, tgt)
        page = src.astype(np.float32)
        hand = mask.astype(bool)
    else:
        # (b) 인쇄 페이지 + 합성 손글씨
        if use_exam and mode < 0.6:
            _, tgt = exam.load(rng, scale * rng.uniform(0.75, 1.0))
            page = _random_crop(rng, [tgt], region)[0].astype(np.float32)
        else:
            page = render_print_page(rng, fonts, region, region).astype(np.float32)
        hand = np.zeros(page.shape[:2], bool)
        n_layers = rng.randint(1, 3)
        for _ in range(n_layers):
            if use_exam and rng.random() < 0.35:
                s2, t2 = exam.load(rng, scale * rng.uniform(0.75, 1.0))
                s2, t2 = _random_crop(rng, [s2, t2], region)
                alpha, _ = ExamInk.ink_from_pair(s2, t2)
            else:
                alpha = render_hand_alpha(rng, fonts, region, region, scale=scale)
            page, visible = apply_ink(page, alpha, rng)
            hand |= visible > 0.07
        page = np.clip(page, 0, 255)

    img = photo_aug(page.astype(np.uint8), rng)
    img = normalize(img)
    x, y = _random_crop(rng, [img, hand.astype(np.uint8)], tile)
    return x, y

In [ ]:
%%writefile make_dataset.py
"""학습용 조각 미리 만들어 두기 (여러 CPU로 병렬).
사용법: python make_dataset.py --exam work/examink --out work/tiles --n 8000
결과: out/x/00001.jpg (입력), out/y/00001.png (손글씨 마스크 0/255)
"""
import argparse
import os
import random
from multiprocessing import Pool, cpu_count

import numpy as np
from PIL import Image

from wn.synth import Fonts, ExamInk, make_sample

_fonts = None
_exam = None
_args = None


def _init(args):
    global _fonts, _exam, _args
    _args = args
    _fonts = Fonts(args.fonts)
    _exam = ExamInk(args.exam) if args.exam else None


def _work(i):
    rng = random.Random(_args.seed * 1_000_003 + i)
    np.random.seed((_args.seed * 7919 + i) % (2**31))
    x, y = make_sample(rng, _fonts, _exam, tile=_args.tile)
    Image.fromarray(x).save(os.path.join(_args.out, 'x', f'{i:06d}.jpg'), quality=95)
    Image.fromarray((y * 255).astype(np.uint8)).save(os.path.join(_args.out, 'y', f'{i:06d}.png'))
    return i


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--exam', default='')
    ap.add_argument('--fonts', default='work/fonts')
    ap.add_argument('--out', default='work/tiles')
    ap.add_argument('--n', type=int, default=8000)
    ap.add_argument('--start', type=int, default=0)
    ap.add_argument('--tile', type=int, default=384)
    ap.add_argument('--seed', type=int, default=1)
    ap.add_argument('--workers', type=int, default=max(1, cpu_count()))
    args = ap.parse_args()
    os.makedirs(os.path.join(args.out, 'x'), exist_ok=True)
    os.makedirs(os.path.join(args.out, 'y'), exist_ok=True)
    Fonts(args.fonts)  # 폰트 미리 받기
    with Pool(args.workers, initializer=_init, initargs=(args,)) as pool:
        for k, _ in enumerate(pool.imap_unordered(_work, range(args.start, args.start + args.n), chunksize=8)):
            if (k + 1) % 200 == 0:
                print(f'{k + 1}/{args.n}', flush=True)
    print('done', args.n)

In [ ]:
%%writefile train.py
"""손글씨 분리 모델 학습 + ONNX 내보내기.
사용법: python train.py --tiles work/tiles --out work/run1 --epochs 30
GPU가 있으면 자동으로 사용 (Colab). 끝나면 out/inkseg.onnx 생성.
"""
import argparse
import glob
import os
import random
import time

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

from wn.model import InkUNet, count_params


class Tiles(Dataset):
    def __init__(self, files, train):
        self.files = files
        self.train = train

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        xp = self.files[i]
        yp = xp.replace(os.sep + 'x' + os.sep, os.sep + 'y' + os.sep).replace('/x/', '/y/')[:-4] + '.png'
        x = np.asarray(Image.open(xp).convert('RGB'), dtype=np.float32) / 255.0
        y = (np.asarray(Image.open(yp)) > 127).astype(np.float32)
        if self.train:
            if random.random() < 0.5:   # 좌우 뒤집기
                x, y = x[:, ::-1], y[:, ::-1]
            # 밝기/대비 살짝
            x = np.clip((x - 0.5) * random.uniform(0.85, 1.15) + 0.5 + random.uniform(-0.04, 0.04), 0, 1)
        x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))
        y = torch.from_numpy(np.ascontiguousarray(y))[None]
        return x, y


def loss_fn(logits, y):
    bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=torch.tensor(2.0, device=y.device))
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)
    return bce + dice.mean()


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    tp = fp = fn = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        p = (torch.sigmoid(model(x)) > 0.5).float()
        tp += (p * y).sum().item()
        fp += (p * (1 - y)).sum().item()
        fn += ((1 - p) * y).sum().item()
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    f1 = 2 * prec * rec / max(1e-9, prec + rec)
    return prec, rec, f1


def export_onnx(model, path):
    model = model.cpu().eval()
    dummy = torch.rand(1, 3, 256, 256)
    torch.onnx.export(
        model, dummy, path, input_names=['image'], output_names=['logits'], opset_version=17,
        dynamic_axes={'image': {2: 'h', 3: 'w'}, 'logits': {2: 'h', 3: 'w'}}, dynamo=False)
    # 검증
    import onnxruntime as ort
    sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    out = sess.run(None, {'image': np.random.rand(1, 3, 320, 480).astype(np.float32)})[0]
    print('onnx ok', out.shape, f'{os.path.getsize(path) / 1e6:.2f} MB')


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--tiles', nargs='+', default=['work/tiles'])
    ap.add_argument('--out', default='work/run1')
    ap.add_argument('--epochs', type=int, default=30)
    ap.add_argument('--batch', type=int, default=16)
    ap.add_argument('--lr', type=float, default=2e-3)
    ap.add_argument('--workers', type=int, default=2)
    ap.add_argument('--resume', default='')
    ap.add_argument('--val_frac', type=float, default=0.05)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    files = sorted(sum([glob.glob(os.path.join(t, 'x', '*.jpg')) for t in args.tiles], []))
    random.Random(0).shuffle(files)
    nval = max(8, int(len(files) * args.val_frac))
    val_files, train_files = files[:nval], files[nval:]
    print(f'train {len(train_files)}  val {len(val_files)}')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = InkUNet().to(device)
    if args.resume:
        model.load_state_dict(torch.load(args.resume, map_location=device))
        print('resumed from', args.resume)
    print(f'params {count_params(model) / 1e6:.2f}M  device {device}')

    tl = DataLoader(Tiles(train_files, True), batch_size=args.batch, shuffle=True, num_workers=args.workers, drop_last=True, pin_memory=device == 'cuda')
    vl = DataLoader(Tiles(val_files, False), batch_size=args.batch, num_workers=args.workers)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    steps = args.epochs * len(tl)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=args.lr, total_steps=max(1, steps), pct_start=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')

    best = -1
    for ep in range(args.epochs):
        model.train()
        t0, tot = time.time(), 0.0
        for x, y in tl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == 'cuda'):
                logits = model(x)
            loss = loss_fn(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item()
        prec, rec, f1 = evaluate(model, vl, device)
        print(f'epoch {ep + 1}/{args.epochs}  loss {tot / max(1, len(tl)):.4f}  '
              f'prec {prec:.3f}  rec {rec:.3f}  F1 {f1:.3f}  ({time.time() - t0:.0f}s)', flush=True)
        if f1 > best:
            best = f1
            torch.save(model.state_dict(), os.path.join(args.out, 'best.pt'))
    torch.save(model.state_dict(), os.path.join(args.out, 'last.pt'))

    model.load_state_dict(torch.load(os.path.join(args.out, 'best.pt'), map_location='cpu'))
    export_onnx(model, os.path.join(args.out, 'inkseg.onnx'))
    print('best F1', round(best, 4))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile eval_photos.py
"""학습된 모델을 실제 사진에 돌려보기 (앱과 같은 순서: 배경 정리 → 모델 → 손글씨 부분 흰색).
사용법: python eval_photos.py --model work/run1/inkseg.onnx --photos ../샘플사진 --out work/eval
결과: 사진마다 [배경 정리본 | 모델이 찾은 손글씨(빨강) | 지운 결과] 3장 나란히.
"""
import argparse
import glob
import os

import numpy as np
import onnxruntime as ort
from PIL import Image, ImageOps
from scipy.ndimage import binary_dilation

from wn.normalize import normalize


def run(sess, norm):
    h, w = norm.shape[:2]
    ph, pw = (-h) % 16, (-w) % 16
    x = np.pad(norm, ((0, ph), (0, pw), (0, 0)), mode='constant', constant_values=255)
    x = x.astype(np.float32).transpose(2, 0, 1)[None] / 255.0
    logits = sess.run(None, {'image': x})[0][0, 0, :h, :w]
    return 1 / (1 + np.exp(-logits))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', required=True)
    ap.add_argument('--photos', default='../샘플사진')
    ap.add_argument('--out', default='work/eval')
    ap.add_argument('--max_side', type=int, default=1400)
    ap.add_argument('--th', type=float, default=0.5)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    sess = ort.InferenceSession(a.model, providers=['CPUExecutionProvider'])
    for p in sorted(glob.glob(os.path.join(a.photos, '*.jpg')) + glob.glob(os.path.join(a.photos, '*.png'))):
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB')  # 폰 회전 정보 반영 (브라우저와 동일)
        s = a.max_side / max(im.size)
        im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
        norm = normalize(np.asarray(im))
        prob = run(sess, norm)
        hand = binary_dilation(prob > a.th, iterations=1)
        over = norm.copy()
        over[hand] = (over[hand] * 0.3 + np.array([255, 0, 0]) * 0.7).astype(np.uint8)
        clean = norm.copy()
        clean[hand] = 255
        gap = np.full((norm.shape[0], 10, 3), 128, np.uint8)
        out = np.concatenate([norm, gap, over, gap, clean], axis=1)
        name = os.path.splitext(os.path.basename(p))[0]
        Image.fromarray(out).save(os.path.join(a.out, f'{name}.jpg'), quality=85)
        print('saved', name)


if __name__ == '__main__':
    main()

In [ ]:
!pip -q install onnx onnxruntime onnxscript huggingface_hub

## 2. 공개 데이터 받기 (ExamInk-Seg, 약 2.5GB · 3~5분)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('ynyg/ExamInk-Seg', repo_type='dataset', local_dir='work/examink',
                  allow_patterns=['data/train/source/*', 'data/train/target/*'])
import os; print(len(os.listdir('work/examink/data/train/source')), '장')

## 3. 학습 조각 만들기 (약 8,000개 · 20~30분)

In [ ]:
!python make_dataset.py --exam work/examink --out work/tiles --n 8000

## 4. 학습 (약 10~20분)

In [ ]:
!python train.py --tiles work/tiles --out work/run1 --epochs 25 --batch 32 --workers 2

## 5. 결과 파일 받기

In [ ]:
from google.colab import files
files.download('work/run1/inkseg.onnx')